# w4_cpu5_dair_o2_external


In [ ]:
import hashlib, io, json, os, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path

WORK = Path('/kaggle/working')
MOUNT = Path('/kaggle/input')
assert WORK.is_dir(), 'Run on Kaggle'
REV = 'de6ecc4f35be0019d368bb033d35ff113f7e6083'
with tempfile.TemporaryDirectory(prefix='w4_dair_') as ts:
    tmp = Path(ts)
    repo = tmp / 'repo'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    env = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1', 'CUDA_VISIBLE_DEVICES': ''}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime'], check=True)
    heldout_bin = next(MOUNT.rglob('shortpose_heldout.bin'))
    pose_npz, meta_dir = tmp / 'pose.npz', tmp / 'pinns_meta'
    with zipfile.ZipFile(heldout_bin) as zf:
        pose_npz.write_bytes(zf.read('pose.npz'))
    subprocess.run([sys.executable, '-m', 'scripts.stage_pinns_metadata', '--output-dir', str(meta_dir)], cwd=repo, env=env, check=True)
    dest = WORK / 'dair_o2_bioeq_external.zip'
    subprocess.run([sys.executable, '-u', '-m', 'scripts.run_dair_o2_bioeq_external', str(meta_dir), str(pose_npz), str(dest)], cwd=repo, env=env, check=True)
    print('DONE:', dest, dest.stat().st_size, hashlib.sha256(dest.read_bytes()).hexdigest(), flush=True)
